# AI Stock Analyst
 This project analyzes stock data, news sentiment, and generates AI insights.

# Features
 - Stock price analysis
 - News sentiment detection
 - AI investment insight using Groq Llama model

<table>
<tr>
<td align="center">

### Workflow Diagram
<img src="flow_Diagram.png" width="450">

</td>

<td align="center">

### Architecture Diagram
<img src="architecture.png" width="600">

</td>
</tr>
</table>

In [1]:
!pip install streamlit yfinance pandas plotly textblob requests langchain-groq


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import yfinance as yf
import pandas as pd
import plotly.express as px
from textblob import TextBlob
import requests
from langchain_groq import ChatGroq

In [3]:
symbol = "AAPL"
groq_key = "YOUR_GROQ_API_KEY"

In [4]:
def fetch_stock_data(symbol):
    
    stock = yf.Ticker(symbol)
    df = stock.history(period="6mo")
    
    return df

df = fetch_stock_data(symbol)

df.tail()

,Open,High,Low,Close,Volume,Dividends,Stock Splits
Date,,,,,,,
2026-02-26 00:00:00-05:00,274.950012,276.109985,270.799988,272.950012,32345100,0.0,0.0
2026-02-27 00:00:00-05:00,272.809998,272.809998,262.890015,264.179993,72366500,0.0,0.0
2026-03-02 00:00:00-05:00,262.410004,266.529999,260.200012,264.720001,41827900,0.0,0.0
2026-03-03 00:00:00-05:00,263.480011,265.559998,260.130005,263.750000,38568900,0.0,0.0
2026-03-04 00:00:00-05:00,264.649994,266.149994,261.420013,262.519989,39751400,0.0,0.0


In [5]:
fig = px.line(
    df,
    x=df.index,
    y="Close",
    title="Stock Price (Last 6 Months)"
)

fig.show()

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

In [ ]:
def sentiment(text):

    analysis = TextBlob(text)

    if analysis.sentiment.polarity > 0:
        return "Positive"
    
    elif analysis.sentiment.polarity < 0:
        return "Negative"
    
    else:
        return "Neutral"

In [ ]:
def get_news(symbol):

    API_KEY = "Your_Api_Key"

    url = f"https://www.alphavantage.co/query?function=NEWS_SENTIMENT&tickers={symbol}&apikey={API_KEY}"

    r = requests.get(url)
    data = r.json()

    articles = []

    if "feed" in data:

        for item in data["feed"][:5]:

            title = item["title"]
            senti = sentiment(title)

            articles.append({
                "Headline": title,
                "Sentiment": senti
            })

    return pd.DataFrame(articles)

news = get_news(symbol)

news

In [ ]:
llm = ChatGroq(
    groq_api_key=groq_key,
    model_name="llama-3.3-70b-versatile"
)

prompt = f"""
Analyze the stock {symbol}.

Provide:
1. Trend summary
2. Investment insight
3. Risk analysis
"""

response = llm.invoke(prompt)

print(response.content)